# RCME-TimeXer: Bitcoin next-day forecasting

This Colab notebook compares **XGBoost, DLinear, LSTM, the original TimeXer, and RCME-TimeXer** using one chronological train/validation/test split.

- **Target:** next-day BTC log return, with an implied next-day closing price.
- **Input:** BTC OHLCV-derived features; these are auxiliary variables, not independent economic exogenous series.
- **RCME:** CEEMDAN is applied separately to each **historical input window**. IMF groups feed a shared, unmodified TimeXer backbone, followed by an adaptive gate and a raw-history skip connection.
- **Evaluation:** 70% train, 15% validation, 15% test by target date. Model selection uses validation only; test predictions are computed after training.
- **Pilot:** FAST_MODE=True reduces the rows, CEEMDAN trials and training epochs. Results in this mode are for checking the pipeline, not for publication.

Run the cells in order. The data file remains unchanged. Outputs and decomposition cache are written to an RCME_TimeXer_outputs subdirectory beside the input CSV.

Original TimeXer: https://github.com/thuml/TimeXer  
CEEMDAN package: https://pypi.org/project/EMD-signal/

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
import sys
from pathlib import Path

def run_command(*parts, cwd=None):
    subprocess.run(list(parts), cwd=cwd, check=True)

run_command(
    sys.executable, "-m", "pip", "install", "-q",
    "EMD-signal==1.10.0",
    "reformer-pytorch==1.4.4",
    "xgboost==3.4.2",
    "einops"
)

REPO = Path("/content/TimeXer")
REPO_COMMIT = "76011909357972bd55a27adba2e1be994d81b327"
if not (REPO / ".git").exists():
    run_command("git", "clone", "-q",
                "https://github.com/thuml/TimeXer.git", str(REPO))
run_command("git", "fetch", "-q", "--depth", "1",
            "origin", REPO_COMMIT, cwd=str(REPO))
run_command("git", "checkout", "-q", REPO_COMMIT, cwd=str(REPO))
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Original TimeXer commit:", REPO_COMMIT)

## Configuration and BTC input

The path supplied by the researcher is tried first. Colab commonly mounts My Drive under `/content/drive/MyDrive`, which is also checked. The end of daily bar \(t\) is the forecast origin; the label is the close-to-close return at \(t+1\). No feature at \(t+1\) enters the input.

In [ ]:
import copy
import hashlib
import json
import math
import time
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PyEMD import CEEMDAN
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBRegressor
from models import DLinear as OriginalDLinear
from models import TimeXer as OriginalTimeXer

SEED = 2027
FAST_MODE = True  # Set to False for the full daily-data experiment.
LOOKBACK = 128
PATCH_LEN = 16
BATCH_SIZE = 32
MAX_ROWS = 1400 if FAST_MODE else None
STEP = 2 if FAST_MODE else 1
CEEMDAN_TRIALS = 4 if FAST_MODE else 20
EPOCHS = 4 if FAST_MODE else 20
PATIENCE = 2 if FAST_MODE else 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

assert LOOKBACK % PATCH_LEN == 0, "LOOKBACK must be divisible by PATCH_LEN."
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

candidates = [
    Path("/content/drive/NCS/Hoa/Data/BTC-USD_all.csv"),
    Path("/content/drive/MyDrive/NCS/Hoa/Data/BTC-USD_all.csv"),
]
DATA_PATH = next((p for p in candidates if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "BTC-USD_all.csv not found. Checked: " +
        ", ".join(str(p) for p in candidates)
    )
OUT_DIR = DATA_PATH.parent / "RCME_TimeXer_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input:", DATA_PATH)
print("Device:", DEVICE)
print("Mode:", "pilot" if FAST_MODE else "full")

In [ ]:
raw = pd.read_csv(DATA_PATH)
lookup = {str(c).strip().lower(): c for c in raw.columns}
required = ["date", "open", "high", "low", "close", "volume"]
missing = [c for c in required if c not in lookup]
if missing:
    raise ValueError(
        f"Missing columns: {missing}. Actual columns: {list(raw.columns)}"
    )

raw = raw.rename(columns={lookup[c]: c for c in required})
raw["date"] = (
    pd.to_datetime(raw["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)
      .dt.normalize()
)
for col in required[1:]:
    raw[col] = pd.to_numeric(
        raw[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )
raw = (raw.dropna(subset=required)
          .sort_values("date")
          .drop_duplicates("date", keep="last")
          .copy())
raw = raw[
    (raw[["open", "high", "low", "close"]] > 0).all(axis=1)
    & (raw["volume"] >= 0)
].copy()

raw["ret"] = np.log(raw["close"]).diff()
raw["hl_range"] = np.log(raw["high"] / raw["low"])
raw["oc_body"] = np.log(raw["close"] / raw["open"])
raw["log_volume"] = np.log1p(raw["volume"])
raw["volume_change"] = raw["log_volume"].diff()
raw["momentum_7"] = np.log(raw["close"] / raw["close"].shift(7))
raw["volatility_7"] = raw["ret"].rolling(7).std()
weekday = raw["date"].dt.dayofweek
raw["weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
raw["weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

AUXILIARY = [
    "hl_range", "oc_body", "log_volume", "volume_change",
    "momentum_7", "volatility_7", "weekday_sin", "weekday_cos"
]
FEATURES = AUXILIARY + ["ret"]  # Target last, as required by TimeXer MS.
df = (raw.replace([np.inf, -np.inf], np.nan)
         .dropna(subset=FEATURES)
         .reset_index(drop=True))
if MAX_ROWS is not None:
    df = df.tail(MAX_ROWS).reset_index(drop=True)
if len(df) < LOOKBACK + 100:
    raise ValueError("Insufficient valid dates after cleaning.")

n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)
scaler = StandardScaler().fit(df.loc[:train_end - 1, FEATURES])
values = scaler.transform(df[FEATURES]).astype(np.float32)
target_mean, target_scale = map(
    float, (scaler.mean_[-1], scaler.scale_[-1])
)

# Exclude windows spanning missing calendar dates, including the target day.
day_number = df["date"].to_numpy(dtype="datetime64[D]").astype(np.int64)
origins = np.array([
    t for t in range(LOOKBACK - 1, n - 1, STEP)
    if np.all(np.diff(day_number[t - LOOKBACK + 1:t + 2]) == 1)
], dtype=np.int64)
if not len(origins):
    raise ValueError("No consecutive-day windows found; inspect the source CSV.")

X = np.stack([
    values[t - LOOKBACK + 1:t + 1] for t in origins
]).astype(np.float32)
Y = values[origins + 1, -1].astype(np.float32)

train_ids = np.where(origins + 1 < train_end)[0]
val_ids = np.where(
    (origins + 1 >= train_end) & (origins + 1 < val_end)
)[0]
test_ids = np.where(origins + 1 >= val_end)[0]
if min(map(len, (train_ids, val_ids, test_ids))) < 15:
    raise ValueError("A split has fewer than 15 windows; increase MAX_ROWS.")

assert max(origins[train_ids] + 1) < train_end
assert min(origins[val_ids] + 1) >= train_end
assert min(origins[test_ids] + 1) >= val_end
assert X.shape == (len(origins), LOOKBACK, len(FEATURES))
assert np.isfinite(X).all() and np.isfinite(Y).all()

close = df["close"].to_numpy(dtype=np.float64)
print("Samples (train/validation/test):",
      len(train_ids), len(val_ids), len(test_ids))
print("Test target dates:",
      df.loc[origins[test_ids[0]] + 1, "date"].date(),
      "to", df.loc[origins[test_ids[-1]] + 1, "date"].date())

## Models and validation-only training

DLinear and TimeXer are instantiated from the pinned official TimeXer repository. Both use its original model code, with a smaller configuration suitable for a Colab pilot. Calendar sine/cosine features are included in the same input tensor for all five models; no model receives future calendar or market observations. The test partition is not evaluated inside training epochs.

In [ ]:
cfg = SimpleNamespace(
    task_name="long_term_forecast",
    features="MS",
    seq_len=LOOKBACK,
    pred_len=1,
    enc_in=X.shape[-1],
    patch_len=PATCH_LEN,
    d_model=64,
    d_ff=128,
    n_heads=4,
    e_layers=1,
    factor=1,
    dropout=0.1,
    activation="gelu",
    embed="timeF",
    freq="d",
    use_norm=0,  # Each feature was fit-scaled on the train period.
    moving_avg=25,
)

class RepoBaseline(nn.Module):
    def __init__(self, name):
        super().__init__()
        cls = OriginalDLinear if name == "DLinear" else OriginalTimeXer
        self.backbone = cls.Model(cfg)

    def forward(self, x):
        return self.backbone(x, None, None, None)[:, 0, -1]

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.LSTM(
            input_size=X.shape[-1], hidden_size=64,
            batch_first=True
        )
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        sequence, _ = self.rnn(x)
        return self.head(sequence[:, -1]).squeeze(-1)

def make_loader(ids, groups=None, shuffle=False):
    tensors = [
        torch.from_numpy(X[ids]),
        torch.from_numpy(Y[ids]),
    ]
    if groups is not None:
        tensors.append(torch.from_numpy(groups[ids]))
    return DataLoader(
        TensorDataset(*tensors),
        batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0
    )

def forward_batch(model, batch, grouped):
    xb = batch[0].to(DEVICE)
    return (
        model(xb, batch[2].to(DEVICE))
        if grouped else model(xb)
    )

def fit_network(model, name, groups=None, learning_rate=5e-4):
    grouped = groups is not None
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate
    )
    train_loader = make_loader(train_ids, groups, shuffle=True)
    validation_loader = make_loader(val_ids, groups)
    best_loss = float("inf")
    best_state = None
    stale_epochs = 0

    for epoch in range(EPOCHS):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            prediction = forward_batch(model, batch, grouped)
            label = batch[1].to(DEVICE)
            loss = F.mse_loss(prediction, label)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        total, count = 0.0, 0
        with torch.no_grad():
            for batch in validation_loader:
                prediction = forward_batch(model, batch, grouped)
                label = batch[1].to(DEVICE)
                total += F.mse_loss(
                    prediction, label, reduction="sum"
                ).item()
                count += label.numel()
        validation_mse = total / count
        print(f"{name}: epoch={epoch + 1} val_MSE={validation_mse:.6f}")

        if validation_mse < best_loss - 1e-7:
            best_loss = validation_mse
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                break

    assert best_state is not None
    model.load_state_dict(best_state)
    return model.cpu()

@torch.no_grad()
def predict_test(model, groups=None):
    model = model.to(DEVICE).eval()
    batches = []
    for batch in make_loader(test_ids, groups):
        batches.append(
            forward_batch(model, batch, groups is not None)
            .detach().cpu().numpy()
        )
    model.cpu()
    return np.concatenate(batches)

# Check that the original modules produce one prediction per input window.
probe = torch.from_numpy(X[:2])
for model_name in ["DLinear", "TimeXer"]:
    trial = RepoBaseline(model_name)
    with torch.no_grad():
        assert trial(probe).shape == (2,)
print("Original DLinear and TimeXer forward shapes: OK")

In [ ]:
models = {}
training_seconds = {}

for name, factory, rate in [
    ("DLinear", lambda: RepoBaseline("DLinear"), 1e-3),
    ("LSTM", LSTMModel, 1e-3),
    ("TimeXer", lambda: RepoBaseline("TimeXer"), 5e-4),
]:
    torch.manual_seed(SEED)
    started = time.perf_counter()
    models[name] = fit_network(
        factory(), name, learning_rate=rate
    )
    training_seconds[name] = time.perf_counter() - started

flat = X.reshape(len(X), -1)
xgboost = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    n_estimators=300,
    max_depth=3,
    learning_rate=0.04,
    subsample=0.9,
    colsample_bytree=0.9,
    early_stopping_rounds=20,
    random_state=SEED,
    n_jobs=2,
)
started = time.perf_counter()
xgboost.fit(
    flat[train_ids], Y[train_ids],
    eval_set=[(flat[val_ids], Y[val_ids])],
    verbose=False,
)
training_seconds["XGBoost"] = time.perf_counter() - started
print("Baseline fitting complete. Test partition has not been evaluated.")

## Rolling CEEMDAN and RCME-TimeXer

For each origin \(t\), decomposition receives only the normalized return window \(t-L+1,\ldots,t\). The first IMF is the fast group, IMFs 2–3 form the medium group, later IMFs form the slow group, and the decomposition residue is retained. Group count stays fixed even if CEEMDAN returns varying numbers of IMFs. The four groups must reconstruct the historical return window. Component-specific future labels are never computed.

CEEMDAN runs on CPU and is the slowest stage. Partial results are saved every 50 windows and reused when the input and decomposition settings match.

In [ ]:
def rolling_ceemdan_groups(windows, window_origins, trials, out_dir):
    signal = np.ascontiguousarray(windows[:, :, -1])
    fingerprint = hashlib.sha256(
        signal.tobytes()
        + f"|trials={trials}|seed={SEED}|max_imf=5".encode()
    ).hexdigest()[:16]
    cache_path = out_dir / f"rolling_ceemdan_{fingerprint}.npz"

    groups = np.zeros(
        (len(windows), 4, LOOKBACK), dtype=np.float32
    )
    completed = np.zeros(len(windows), dtype=bool)
    if cache_path.exists():
        with np.load(cache_path) as saved:
            if (
                saved["groups"].shape == groups.shape
                and saved["completed"].shape == completed.shape
            ):
                groups = saved["groups"]
                completed = saved["completed"]

    ceemdan = CEEMDAN(
        trials=trials, epsilon=0.005, parallel=False
    )
    for j, origin in enumerate(window_origins):
        if completed[j]:
            continue
        history = signal[j].astype(np.float64)

        if np.std(history) < 1e-10:
            groups[j, 3] = history
        else:
            ceemdan.noise_seed(SEED + int(origin))
            ceemdan.ceemdan(history, max_imf=5)
            imfs, residue = ceemdan.get_imfs_and_residue()
            imfs = np.atleast_2d(imfs)
            groups[j, 0] = imfs[0]
            if len(imfs) > 1:
                groups[j, 1] = imfs[1:3].sum(axis=0)
            if len(imfs) > 3:
                groups[j, 2] = imfs[3:].sum(axis=0)
            groups[j, 3] = residue

        if not np.allclose(
            groups[j].sum(axis=0), history, atol=2e-4
        ):
            raise AssertionError(
                f"Input reconstruction failed at origin {origin}"
            )
        completed[j] = True
        if (j + 1) % 50 == 0:
            np.savez_compressed(
                cache_path, groups=groups, completed=completed
            )
            print(f"CEEMDAN: {j + 1}/{len(windows)} windows")

    np.savez_compressed(
        cache_path, groups=groups, completed=completed
    )
    assert completed.all()
    assert np.allclose(
        groups.sum(axis=1), signal, atol=2e-4
    )
    print("CEEMDAN cache:", cache_path)
    return groups

started = time.perf_counter()
GROUPS = rolling_ceemdan_groups(
    X, origins, CEEMDAN_TRIALS, OUT_DIR
)
decomposition_seconds = time.perf_counter() - started
print("Decomposition seconds:", round(decomposition_seconds, 1))

In [ ]:
class RCMETimeXer(nn.Module):
    """
    Shared original TimeXer backbone for four rolling CEEMDAN groups.
    Each call has the same BTC auxiliary variables and a different
    historical return component as its final/target channel.
    """
    def __init__(self):
        super().__init__()
        self.shared_timexer = OriginalTimeXer.Model(cfg)
        self.gate = nn.Sequential(
            nn.Linear(8 + X.shape[-1], 32),
            nn.ReLU(),
            nn.Linear(32, 4),
        )
        self.raw_skip = nn.Linear(LOOKBACK, 1)

    def forward(self, x, groups):
        forecasts = []
        for k in range(4):
            component_input = torch.cat(
                [x[:, :, :-1], groups[:, k, :].unsqueeze(-1)],
                dim=-1,
            )
            component_forecast = self.shared_timexer(
                component_input, None, None, None
            )[:, 0, 0]
            forecasts.append(component_forecast)

        forecasts = torch.stack(forecasts, dim=1)
        gate_input = torch.cat(
            [
                groups.mean(dim=-1),
                groups.std(dim=-1, unbiased=False),
                x[:, -1, :],
            ],
            dim=1,
        )
        weights = torch.softmax(self.gate(gate_input), dim=1)
        return (
            (weights * forecasts).sum(dim=1)
            + self.raw_skip(x[:, :, -1]).squeeze(-1)
        )

torch.manual_seed(SEED)
rcme = RCMETimeXer()
with torch.no_grad():
    trial_output = rcme(
        torch.from_numpy(X[:2]),
        torch.from_numpy(GROUPS[:2]),
    )
assert trial_output.shape == (2,) and torch.isfinite(trial_output).all()
print("RCME-TimeXer forward shape: OK")

started = time.perf_counter()
models["RCME-TimeXer"] = fit_network(
    rcme, "RCME-TimeXer",
    groups=GROUPS,
    learning_rate=5e-4,
)
training_seconds["RCME-TimeXer"] = time.perf_counter() - started

## One final test evaluation and exported artifacts

All five models are now selected by validation. The test cell below reports return MAE/RMSE, directional accuracy and implied closing-price MAE. It writes predictions, metrics, a run manifest and a figure beside the input CSV. RMSE on near-zero returns can differ substantially from price-level RMSE; use both alongside a naïve return-zero benchmark in a later full study.

In [ ]:
predictions_standardized = {
    "XGBoost": xgboost.predict(flat[test_ids]),
}
for name, model in models.items():
    predictions_standardized[name] = predict_test(
        model,
        GROUPS if name == "RCME-TimeXer" else None,
    )

def inverse_return(z):
    return np.asarray(z) * target_scale + target_mean

true_return = inverse_return(Y[test_ids])
previous_close = close[origins[test_ids]]
true_close = close[origins[test_ids] + 1]
test_dates = df.loc[
    origins[test_ids] + 1, "date"
].to_numpy()

metrics_rows = []
prediction_table = pd.DataFrame({
    "date": test_dates,
    "previous_close": previous_close,
    "actual_return": true_return,
    "actual_close": true_close,
})
for name, predicted_standardized in predictions_standardized.items():
    estimated_return = inverse_return(predicted_standardized)
    assert np.isfinite(estimated_return).all(), name
    estimated_close = previous_close * np.exp(estimated_return)

    metrics_rows.append({
        "model": name,
        "return_MAE": mean_absolute_error(
            true_return, estimated_return
        ),
        "return_RMSE": math.sqrt(mean_squared_error(
            true_return, estimated_return
        )),
        "direction_accuracy": float(np.mean(
            (estimated_return > 0) == (true_return > 0)
        )),
        "close_MAE_USD": mean_absolute_error(
            true_close, estimated_close
        ),
        "training_seconds": training_seconds[name],
    })
    prediction_table[f"{name}_return"] = estimated_return
    prediction_table[f"{name}_close"] = estimated_close

metrics_table = (
    pd.DataFrame(metrics_rows)
      .sort_values("return_RMSE")
      .reset_index(drop=True)
)
display(metrics_table)

metrics_table.to_csv(
    OUT_DIR / "test_metrics.csv", index=False
)
prediction_table.to_csv(
    OUT_DIR / "test_predictions.csv", index=False
)
manifest = {
    "source_path": str(DATA_PATH),
    "source_sha256": hashlib.sha256(
        DATA_PATH.read_bytes()
    ).hexdigest(),
    "original_timexer_commit": REPO_COMMIT,
    "fast_mode": FAST_MODE,
    "seed": SEED,
    "lookback": LOOKBACK,
    "patch_len": PATCH_LEN,
    "forecast_horizon": 1,
    "window_step": STEP,
    "ceemdan_trials": CEEMDAN_TRIALS,
    "ceemdan_max_imf": 5,
    "chronological_split": [0.70, 0.15, 0.15],
    "input_columns": FEATURES,
    "train_samples": len(train_ids),
    "validation_samples": len(val_ids),
    "test_samples": len(test_ids),
    "device": DEVICE,
    "decomposition_seconds": decomposition_seconds,
}
(OUT_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(
    metrics_table["model"],
    metrics_table["return_RMSE"],
)
ax.set_ylabel("Test RMSE of next-day BTC log return")
ax.tick_params(axis="x", labelrotation=25)
fig.tight_layout()
fig.savefig(OUT_DIR / "test_return_rmse.png", dpi=180)
plt.show()

for model_name, trained_model in models.items():
    torch.save(
        trained_model.state_dict(),
        OUT_DIR / (model_name.replace("-", "_") + "_state.pt"),
    )
xgboost.save_model(str(OUT_DIR / "XGBoost.json"))

print("Saved outputs to:", OUT_DIR)
print("Pilot results are for workflow verification only." if FAST_MODE
      else "Full run completed; statistical validation is a separate step.")

## Interpretation and next step

This notebook checks the five requested models on one BTC dataset. FAST_MODE changes the sample length and prediction-origin stride, so it must not be used for a journal claim. For a publication analysis, rerun with FAST_MODE=False, several preset seeds and time periods, and compare an added no-decomposition RCME ablation under the same information set and tuning budget. The notebook has not been executed against the researcher's Drive file during authoring.